# QuantAlpha AI — Step 8: Improved Model (Sentiment + Momentum + Relative Return Target)

Step 7's XGBoost showed no real edge (AUC 0.509, worse than trivial baseline). This notebook
tries 3 legitimate, standard-practice improvements before concluding further:

1. **Relative return target** — predict whether a stock BEATS the Nifty 200 average return over
   the holding period, not just whether it goes up in absolute terms. This strips out market-wide
   noise (if the whole market rallies, most stocks go up regardless of merit) and isolates real
   stock-picking signal — a standard technique in quant research.
2. **Sentiment/event features** — incorporate the FinBERT sentiment + event classification data
   from Step 2B, which was built but never actually used as a model feature until now.
3. **Momentum features across multiple lookback windows** (5, 20, 50 days) instead of a single
   snapshot — gives the model a sense of trajectory, not just a point-in-time reading.

**Honest framing:** these are real, standard improvements — not guaranteed fixes. If this still
shows no edge, that's a strong, final signal that short-term prediction genuinely isn't
achievable with this data/approach, and effort should go toward the Long-Term fundamental signal
instead (which already showed genuine promise).


## 1. Connect to Drive + load everything

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/QuantAlpha')

import pandas as pd
import numpy as np
import glob
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

fundamental_scores = pd.read_csv('data/fundamental_scores.csv')
fundamental_scores['symbol_short'] = fundamental_scores['symbol'].str.replace('.NS', '', regex=False)
fscore_lookup = dict(zip(fundamental_scores['symbol_short'], fundamental_scores['piotroski_f_score']))
roce_lookup = dict(zip(fundamental_scores['symbol_short'], fundamental_scores['roce']))

event_classification = pd.read_csv('data/event_classification.csv') if os.path.exists('data/event_classification.csv') else pd.DataFrame()
print("Data loaded.")


Mounted at /content/drive
Data loaded.


## 2. Build sentiment feature per stock (static — most recent signal we have)
**Honest limitation:** our event data is a CURRENT snapshot (news at time of Step 2B), not
historical daily sentiment. So this feature is the same value for every historical date for a
given stock — it can only help the model learn "stocks currently in the negative-news set behave
differently," not true time-varying sentiment. A fully rigorous version would need historical
daily news sentiment, which is a bigger future data engineering task.


In [2]:
if len(event_classification):
    event_classification['symbol_short'] = event_classification['symbol'].str.replace('.NS', '', regex=False)
    sentiment_lookup = event_classification.groupby('symbol_short').agg(
        has_structural_flag=('verdict', lambda x: any('STRUCTURAL' in v for v in x)),
        negative_news_count=('headline', 'count')
    ).to_dict('index')
else:
    sentiment_lookup = {}

print(f"Sentiment data available for {len(sentiment_lookup)} stocks")


Sentiment data available for 74 stocks


## 3. Build training data with momentum features + relative-return target

In [3]:
technical_10yr_files = glob.glob('data_10yr/technical/*.parquet')
symbols_10yr = sorted(f.split('/')[-1].replace('.parquet', '') for f in technical_10yr_files)
print(f"Building improved training data from {len(symbols_10yr)} stocks")

HOLDING_DAYS = 15

# First pass: load all price data and compute market-wide average return per date (proxy for Nifty 200 index)
all_price_data = {}
for sym in symbols_10yr:
    df = pd.read_parquet(f"data_10yr/technical/{sym}.parquet")
    df['Date'] = pd.to_datetime(df['Date']).dt.tz_localize(None)
    df = df.reset_index(drop=True)
    if len(df) > 300:
        all_price_data[sym] = df

print(f"Loaded {len(all_price_data)} stocks for feature building")


Building improved training data from 188 stocks
Loaded 188 stocks for feature building


In [4]:
def get_momentum_features(df, i):
    feats = {}
    for window in [5, 20, 50]:
        if i - window >= 0:
            past_price = df.loc[i - window, 'Close']
            curr_price = df.loc[i, 'Close']
            feats[f'momentum_{window}d'] = (curr_price - past_price) / past_price if past_price != 0 else np.nan
        else:
            feats[f'momentum_{window}d'] = np.nan
    return feats

all_rows = []
for sym, df in all_price_data.items():
    fscore = fscore_lookup.get(sym, np.nan)
    roce = roce_lookup.get(sym, np.nan)
    sentiment_info = sentiment_lookup.get(sym, {})
    has_structural_flag = sentiment_info.get('has_structural_flag', False)
    negative_news_count = sentiment_info.get('negative_news_count', 0)

    for i in range(200, len(df) - HOLDING_DAYS):
        row = {
            'symbol': sym,
            'date': df.loc[i, 'Date'],
            'rsi_14': df.loc[i, 'RSI_14'],
            'adx_14': df.loc[i, 'ADX_14'],
            'atr_14': df.loc[i, 'ATR_14'],
            'close_to_sma50': (df.loc[i, 'Close'] - df.loc[i, 'SMA_50']) / df.loc[i, 'SMA_50'] if pd.notna(df.loc[i, 'SMA_50']) and df.loc[i, 'SMA_50'] != 0 else np.nan,
            'close_to_sma200': (df.loc[i, 'Close'] - df.loc[i, 'SMA_200']) / df.loc[i, 'SMA_200'] if pd.notna(df.loc[i, 'SMA_200']) and df.loc[i, 'SMA_200'] != 0 else np.nan,
            'piotroski_f_score': fscore,
            'roce': roce,
            'has_structural_flag': int(has_structural_flag),
            'negative_news_count': negative_news_count,
        }
        row.update(get_momentum_features(df, i))

        entry_price = df.loc[i, 'Close']
        exit_price = df.loc[i + HOLDING_DAYS, 'Close']
        row['stock_fwd_return'] = (exit_price - entry_price) / entry_price
        row['entry_date_idx'] = i
        all_rows.append(row)

data = pd.DataFrame(all_rows)
print(f"Raw dataset built: {len(data)} rows")


Raw dataset built: 397238 rows


## 4. Compute market-average return per date (proxy index) and the RELATIVE target
For each date, the average forward return across ALL stocks approximates the Nifty 200 index
move over that same period. The target becomes: did THIS stock beat that average?


In [5]:
market_avg_by_date = data.groupby('date')['stock_fwd_return'].transform('mean')
data['market_fwd_return'] = market_avg_by_date
data['excess_return'] = data['stock_fwd_return'] - data['market_fwd_return']
data['target_relative'] = (data['excess_return'] > 0).astype(int)

data = data.dropna(subset=['rsi_14', 'adx_14', 'atr_14', 'close_to_sma50', 'close_to_sma200',
                            'momentum_5d', 'momentum_20d', 'momentum_50d'])
print(f"Final dataset: {len(data)} rows")
print(f"Relative target balance: {data['target_relative'].value_counts(normalize=True).round(3).to_dict()}")


Final dataset: 397238 rows
Relative target balance: {0: 0.542, 1: 0.458}


## 5. Time-based split + train

In [6]:
FEATURES = ['rsi_14', 'adx_14', 'atr_14', 'close_to_sma50', 'close_to_sma200',
            'piotroski_f_score', 'roce', 'has_structural_flag', 'negative_news_count',
            'momentum_5d', 'momentum_20d', 'momentum_50d']

train = data[data['date'] < '2024-01-01'].copy()
test = data[data['date'] >= '2024-01-01'].copy()

for col in ['piotroski_f_score', 'roce']:
    median_val = train[col].median()
    train[col] = train[col].fillna(median_val)
    test[col] = test[col].fillna(median_val)

print(f"Train: {len(train)} rows | Test: {len(test)} rows")

model = xgb.XGBClassifier(
    max_depth=4, n_estimators=300, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, eval_metric='logloss', random_state=42
)
model.fit(train[FEATURES], train['target_relative'])
print("Model trained on RELATIVE (excess-return) target.")


Train: 283160 rows | Test: 114078 rows
Model trained on RELATIVE (excess-return) target.


## 6. Evaluate on unseen 2024-2026 data

In [7]:
preds = model.predict(test[FEATURES])
probs = model.predict_proba(test[FEATURES])[:, 1]

accuracy = accuracy_score(test['target_relative'], preds)
auc = roc_auc_score(test['target_relative'], probs)

print(f"=== Improved Model Results (relative return target) — UNSEEN 2024-2026 ===\n")
print(f"Accuracy: {accuracy*100:.2f}%")
print(f"AUC-ROC: {auc:.3f}")
print(f"\n{classification_report(test['target_relative'], preds, target_names=['UNDERPERFORM', 'OUTPERFORM'])}")

majority_baseline = max(test['target_relative'].mean(), 1 - test['target_relative'].mean())
print(f"\nMajority baseline: {majority_baseline*100:.2f}%")
print(f"Model accuracy:    {accuracy*100:.2f}%")
print(f"Real edge:         {(accuracy-majority_baseline)*100:+.2f} points")

print("\nFeature importance:")
importance = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
print(importance)


=== Improved Model Results (relative return target) — UNSEEN 2024-2026 ===

Accuracy: 53.20%
AUC-ROC: 0.509

              precision    recall  f1-score   support

UNDERPERFORM       0.54      0.89      0.67     61173
  OUTPERFORM       0.48      0.12      0.20     52905

    accuracy                           0.53    114078
   macro avg       0.51      0.50      0.43    114078
weighted avg       0.51      0.53      0.45    114078


Majority baseline: 53.62%
Model accuracy:    53.20%
Real edge:         -0.43 points

Feature importance:
atr_14                 0.096818
roce                   0.096563
negative_news_count    0.094410
close_to_sma200        0.093013
close_to_sma50         0.089016
piotroski_f_score      0.084237
momentum_50d           0.083180
has_structural_flag    0.080337
adx_14                 0.079318
rsi_14                 0.075583
momentum_20d           0.068557
momentum_5d            0.058968
dtype: float32


## 7. How to read this honestly

- **Compare this AUC directly to Step 7's 0.509.** If this is meaningfully higher (0.55+), the
  relative-return framing + extra features genuinely helped — a real, legitimate improvement.
  If it's still ~0.50-0.51, that's strong confirmation that short-term stock-picking alpha isn't
  extractable from this feature set, regardless of framing.
- **Check feature importance** — if `momentum_*` or sentiment features rank highly THIS time
  (unlike Step 7 where nothing stood out), that tells us which specific addition helped.
- Either way, this is close to the limit of what's reasonable to try with free data and daily
  (not intraday) granularity. A further improvement would need intraday data, options flow, or
  analyst estimate revisions — beyond free-tier scope.


## 8. Summary
Whatever this shows, you now have a rigorous, multi-attempt record:
- Rule-based technical (no edge, multiple market regimes)
- Sector momentum (no edge, properly time-split)
- ML on absolute return (no edge, AUC 0.509)
- ML on relative return + sentiment + momentum (this result)
- Fundamental quality, 6-12 month horizon (real edge — your platform's genuine strength)

**This IS a complete, honest research process** — worth documenting as-is in your GitHub README,
regardless of whether this final attempt improved things. Testing multiple legitimate approaches
and reporting results honestly is the actual skill being demonstrated here.
